In [1]:

import sys
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split

from pathlib import Path

sys.path.append(str(Path().resolve().parent / 'src'))
from utils import (backtest_dca_plus_trading_WITH_SL, 
                backtest_dca_plus_trading_NO_SL, 
                backtest_with_atr_SL,
                evaluate_all_strategies,
                backtest_dca_self_sufficient,
                backtest_dca_pure)


In [2]:
#Cargar modelo y scaler.
with open('../models/best_models/xgboost_best_model.pkl', 'rb') as f: 
    ml_model = pickle.load(f)
with open('../models/models_num/scaler_num.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [3]:
#cargar dataframe de SP500
df=pd.read_csv('../data/VOO_ind_signal.csv')

In [4]:
features = [
    'rsi', 'macd_diff', 'bollinger_width', 'atr', 'adx', 
    'volume_ratio', 'price_vs_kijun', 'tenkan_vs_kijun'
]

In [5]:
X_final = df[features]
y_final = df['target']

In [6]:
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(X_final, y_final, test_size=0.2, shuffle=False)

In [7]:
X_final_scaled = scaler.transform(X_final)
df['signal_ml'] = ml_model.predict(X_final_scaled)

### 1.Backtesting del modelo en los datos de test. 

In [8]:
# Solo se cogen los datos de test, excluyendo el entrenamiento. 

df_backtest_test_only = df.loc[X_test_final.index]

In [9]:
start_date = df_backtest_test_only['date'].min()
end_date = df_backtest_test_only['date'].max()

print(f"Fechas del conjunto de test: {start_date} a {end_date}")

Fechas del conjunto de test: 2022-07-25 a 2025-05-02


> **Nota:** El conjunto de test abarca casi tres años: desde el 25 de julio de 2022 hasta el 2 de mayo de 2025.


In [10]:
#Parametros para los backtests 

estrategias_a_probar = [
    # Individuales
    'signal_ema_price',
    'signal_macd_buy',
    'signal_stochastic_buy',
    'signal_ichimoku_kijun_cross',
    'signal_bollinger_buy',
    
    # Combinadas
    'signal_macd_&_stochastic',
    'signal_ema_price_&_macd',
    'signal_ichimoku_kijun_cross_&_stochastic',
    'signal_ema_price_&_stochastic',
    
    #Machine Learning
    'signal_ml'    
]

params = {
    'monthly_invest': 400,
    'trade_amount': 150,
    'take_profit_pct': 0.125,
    'initial_trading_cash': 1000,
    'verbose': False 
}

params_atr = {
    'monthly_invest': 400,
    'trade_amount': 150,
    'atr_multiplier_tp': 10.0,   
    'atr_multiplier_sl': 3.0,  
    'initial_trading_cash': 1000,
    'verbose': False
}

In [11]:
resultados_dca, df_dca = backtest_dca_pure(df_backtest_test_only, monthly_invest=400)

 Resultados para la estrategia: DCA Puro 
Valor Final del Portafolio: $17,340.99
Ganancia/Pérdida Absoluta: $3,340.99
Total Aportado: $14,000.00
Retorno Porcentual: 23.86%



In [12]:
print(" Iniciando Evaluación con Stops sin ATR")
df_resultados_finales = evaluate_all_strategies(df_backtest_test_only, estrategias_a_probar, backtest_dca_plus_trading_WITH_SL,params)

print("\n Tabla Comparativa de Estrategias")
display(df_resultados_finales)

 Iniciando Evaluación con Stops sin ATR
Evaluando: signal_ema_price...
Evaluando: signal_macd_buy...
Evaluando: signal_stochastic_buy...
Evaluando: signal_ichimoku_kijun_cross...
Evaluando: signal_bollinger_buy...
Evaluando: signal_macd_&_stochastic...
Evaluando: signal_ema_price_&_macd...
Evaluando: signal_ichimoku_kijun_cross_&_stochastic...
Evaluando: signal_ema_price_&_stochastic...
Evaluando: signal_ml...

 Tabla Comparativa de Estrategias


,Strategy,Final Portfolio Value,Total Contributions,Absolute Return,Percentage Return,Trading PnL,Trading Capital Used,Open Trades at End,Value of Open Trades,Sharpe Ratio,Total DCA Shares
0,signal_ml (With SL),18742.568,15000,3742.568,24.950,306.797,10200,7,1144.778,2.723,33.405
1,signal_ichimoku_kijun_cross (With SL),18496.201,15000,3496.201,23.308,137.255,3750,2,317.953,2.744,33.405
2,signal_macd_buy (With SL),18419.947,15000,3419.947,22.800,78.955,1050,0,0.000,2.750,33.405
3,signal_ema_price (With SL),18381.251,15000,3381.251,22.542,40.259,750,0,0.000,2.747,33.405
4,signal_stochastic_buy (With SL),18349.764,15000,3349.764,22.332,8.772,900,0,0.000,2.745,33.405
5,signal_bollinger_buy (With SL),18340.992,15000,3340.992,22.273,0.000,0,0,0.000,2.747,33.405
6,signal_macd_&_stochastic (With SL),18340.992,15000,3340.992,22.273,0.000,0,0,0.000,2.747,33.405
7,signal_ema_price_&_macd (With SL),18340.992,15000,3340.992,22.273,0.000,0,0,0.000,2.747,33.405
8,signal_ema_price_&_stochastic (With SL),18340.992,15000,3340.992,22.273,0.000,0,0,0.000,2.747,33.405
9,signal_ichimoku_kijun_cross_&_stochastic (With...,18333.269,15000,3333.269,22.222,-7.723,150,0,0.000,2.746,33.405


In [13]:
print(" Iniciando Evaluación con Stops con ATR")

df_results_atr = evaluate_all_strategies(df_backtest_test_only, estrategias_a_probar,  backtest_with_atr_SL, params_atr)
print("\n Tabla Comparativa con Stops con ATR ")
display(df_results_atr)

 Iniciando Evaluación con Stops con ATR
Evaluando: signal_ema_price...
Evaluando: signal_macd_buy...
Evaluando: signal_stochastic_buy...
Evaluando: signal_ichimoku_kijun_cross...
Evaluando: signal_bollinger_buy...
Evaluando: signal_macd_&_stochastic...
Evaluando: signal_ema_price_&_macd...
Evaluando: signal_ichimoku_kijun_cross_&_stochastic...
Evaluando: signal_ema_price_&_stochastic...
Evaluando: signal_ml...

 Tabla Comparativa con Stops con ATR 


,Strategy,Final Portfolio Value,Total Contributions,Absolute Return,Percentage Return,Trading PnL,Trading Capital Used,Open Trades at End,Value of Open Trades,Sharpe Ratio,Total DCA Shares
0,signal_ml (ATR Stops),18753.772,15000,3753.772,25.025,296.993,9900,8,1315.787,2.717,33.405
1,signal_ichimoku_kijun_cross (ATR Stops),18508.060,15000,3508.060,23.387,142.621,3600,3,474.447,2.744,33.405
2,signal_ema_price (ATR Stops),18414.562,15000,3414.562,22.764,73.569,750,0,0.000,2.748,33.405
3,signal_macd_buy (ATR Stops),18411.478,15000,3411.478,22.743,70.486,1050,0,0.000,2.749,33.405
4,signal_stochastic_buy (ATR Stops),18352.167,15000,3352.167,22.348,11.175,900,0,0.000,2.745,33.405
5,signal_bollinger_buy (ATR Stops),18340.992,15000,3340.992,22.273,0.000,0,0,0.000,2.747,33.405
6,signal_macd_&_stochastic (ATR Stops),18340.992,15000,3340.992,22.273,0.000,0,0,0.000,2.747,33.405
7,signal_ema_price_&_macd (ATR Stops),18340.992,15000,3340.992,22.273,0.000,0,0,0.000,2.747,33.405
8,signal_ema_price_&_stochastic (ATR Stops),18340.992,15000,3340.992,22.273,0.000,0,0,0.000,2.747,33.405
9,signal_ichimoku_kijun_cross_&_stochastic (ATR ...,18336.153,15000,3336.153,22.241,-4.840,150,0,0.000,2.747,33.405


In [14]:
print(" Iniciando Evaluación  sin Stops ni ATR")
df_resultados_finales = evaluate_all_strategies(df_backtest_test_only, estrategias_a_probar, backtest_dca_plus_trading_NO_SL,params)

print("\n Tabla Comparativa de Estrategias sin Stop ")
display(df_resultados_finales)

 Iniciando Evaluación  sin Stops ni ATR
Evaluando: signal_ema_price...
Evaluando: signal_macd_buy...
Evaluando: signal_stochastic_buy...
Evaluando: signal_ichimoku_kijun_cross...
Evaluando: signal_bollinger_buy...
Evaluando: signal_macd_&_stochastic...
Evaluando: signal_ema_price_&_macd...
Evaluando: signal_ichimoku_kijun_cross_&_stochastic...
Evaluando: signal_ema_price_&_stochastic...
Evaluando: signal_ml...

 Tabla Comparativa de Estrategias sin Stop 


,Strategy,Final Portfolio Value,Total Contributions,Absolute Return,Percentage Return,Trading PnL,Trading Capital Used,Open Trades at End,Value of Open Trades,Sharpe Ratio,Total DCA Shares
0,signal_ml (No SL),18800.869,15000,3800.869,25.339,499.150,5250,9,1310.727,2.700,33.405
1,signal_ichimoku_kijun_cross (No SL),18657.308,15000,3657.308,24.382,308.654,3600,8,1207.662,2.740,33.405
2,signal_stochastic_buy (No SL),18457.587,15000,3457.587,23.051,116.595,900,0,0.000,2.750,33.405
3,signal_macd_buy (No SL),18449.293,15000,3449.293,22.995,115.656,1050,1,142.644,2.751,33.405
4,signal_ema_price (No SL),18415.288,15000,3415.288,22.769,76.244,750,1,148.052,2.749,33.405
5,signal_ichimoku_kijun_cross_&_stochastic (No SL),18361.352,15000,3361.352,22.409,20.360,150,0,0.000,2.748,33.405
6,signal_macd_&_stochastic (No SL),18340.992,15000,3340.992,22.273,0.000,0,0,0.000,2.747,33.405
7,signal_bollinger_buy (No SL),18340.992,15000,3340.992,22.273,0.000,0,0,0.000,2.747,33.405
8,signal_ema_price_&_macd (No SL),18340.992,15000,3340.992,22.273,0.000,0,0,0.000,2.747,33.405
9,signal_ema_price_&_stochastic (No SL),18340.992,15000,3340.992,22.273,0.000,0,0,0.000,2.747,33.405


In [15]:
# Deshabilitar gráficos
params['plot'] = False
params['initial_trading_cash']=5000

df_resultados_finales = evaluate_all_strategies(df, estrategias_a_probar, backtest_dca_self_sufficient, params)

print("\n Tabla Comparativa de Estrategias selfsuficcient")
display(df_resultados_finales)

Evaluando: signal_ema_price...
Evaluando: signal_macd_buy...
Evaluando: signal_stochastic_buy...
Evaluando: signal_ichimoku_kijun_cross...
Evaluando: signal_bollinger_buy...
Evaluando: signal_macd_&_stochastic...
Evaluando: signal_ema_price_&_macd...
Evaluando: signal_ichimoku_kijun_cross_&_stochastic...
Evaluando: signal_ema_price_&_stochastic...
Evaluando: signal_ml...

 Tabla Comparativa de Estrategias selfsuficcient


,Strategy,Final Portfolio Value,Total Contributions,Absolute Return,Percentage Return,Trading PnL,Trading Capital Used,Open Trades at End,Value of Open Trades,Sharpe Ratio,DCA Total Shares,DCA Shares Value
0,signal_ml (Self-Sufficient),30457.313,10600,19857.313,187.333,710.024,6450,0,0,0.746,58.461,30347.289
1,signal_macd_buy (Self-Sufficient),27381.011,9800,17581.011,179.398,130.497,2550,0,0,0.711,52.110,27050.515
2,signal_stochastic_buy (Self-Sufficient),27312.522,9800,17512.522,178.699,62.007,2400,0,0,0.709,52.110,27050.515
3,signal_ema_price (Self-Sufficient),27286.531,9800,17486.531,178.434,36.017,2700,0,0,0.708,52.110,27050.515
4,signal_ema_price_&_macd (Self-Sufficient),27269.611,9800,17469.611,178.261,19.097,150,0,0,0.710,52.110,27050.515
5,signal_ichimoku_kijun_cross_&_stochastic (Self...,27256.397,9800,17456.397,178.127,5.882,1350,0,0,0.708,52.110,27050.515
6,signal_bollinger_buy (Self-Sufficient),27250.515,9800,17450.515,178.066,0.000,0,0,0,0.709,52.110,27050.515
7,signal_macd_&_stochastic (Self-Sufficient),27250.515,9800,17450.515,178.066,0.000,0,0,0,0.709,52.110,27050.515
8,signal_ema_price_&_stochastic (Self-Sufficient),27250.515,9800,17450.515,178.066,0.000,0,0,0,0.709,52.110,27050.515
9,signal_ichimoku_kijun_cross (Self-Sufficient),27765.365,10200,17565.365,172.209,244.355,7200,0,0,0.715,53.401,27721.011
